# 04 — Conformal Prediction & Uncertainty Quantification

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

from src.data.split import load_splits
from src.models.nam import NAM
from src.models.nam_trainer import NAMTrainer
from src.models.xgboost_baseline import train_xgboost
from src.conformal.wrapper import NAMSklearnWrapper, create_conformal_classifier, predict_with_confidence
from src.conformal.calibration import compute_all_calibration_metrics, conformal_coverage_analysis
from src.visualization.calibration_plots import plot_reliability_comparison, plot_conformal_set_sizes

## 1. Load Data & Train Models

In [ ]:
splits = load_splits(config['data']['processed_dir'])
X_train = splits['X_train'].values
X_cal = splits['X_cal'].values
X_test = splits['X_test'].values
y_train, y_cal, y_test = splits['y_train'], splits['y_cal'], splits['y_test']
feature_names = splits['feature_names']

# Train NAM
trainer = NAMTrainer(device='cpu')
nam_model, _ = trainer.train_model(X_train, y_train, X_cal, y_cal, config['nam'])

# Train XGBoost
xgb_model = train_xgboost(X_train, y_train, X_cal, y_cal, config['xgboost'])

## 2. Conformal Prediction

In [ ]:
alpha_levels = config['conformal']['alpha_levels']

# NAM conformal
nam_wrapper = NAMSklearnWrapper(nam_model)
nam_wrapper.fit(X_cal, y_cal)
nam_conformal = create_conformal_classifier(nam_wrapper, X_cal, y_cal)
nam_results = predict_with_confidence(nam_conformal, X_test, alpha_levels)

# XGBoost conformal
xgb_conformal = create_conformal_classifier(xgb_model, X_cal, y_cal)
xgb_results = predict_with_confidence(xgb_conformal, X_test, alpha_levels)

## 3. Coverage Analysis

In [ ]:
nam_coverage = conformal_coverage_analysis(y_test, nam_results['y_sets'], alpha_levels)
xgb_coverage = conformal_coverage_analysis(y_test, xgb_results['y_sets'], alpha_levels)

print('NAM Conformal Coverage:')
for c in nam_coverage:
    print(f"  alpha={c['alpha']}: coverage={c['empirical_coverage']:.4f} "
          f"(nominal={c['nominal_coverage']:.2f}), avg_set_size={c['avg_set_size']:.3f}")

print('\nXGBoost Conformal Coverage:')
for c in xgb_coverage:
    print(f"  alpha={c['alpha']}: coverage={c['empirical_coverage']:.4f} "
          f"(nominal={c['nominal_coverage']:.2f}), avg_set_size={c['avg_set_size']:.3f}")

## 4. Calibration Plots

In [ ]:
# Get probabilities
nam_prob = nam_wrapper.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

nam_cal = compute_all_calibration_metrics(y_test, nam_prob)
xgb_cal = compute_all_calibration_metrics(y_test, xgb_prob)

print(f"NAM  — ECE: {nam_cal['ece']:.4f}, Brier: {nam_cal['brier_score']:.4f}")
print(f"XGB  — ECE: {xgb_cal['ece']:.4f}, Brier: {xgb_cal['brier_score']:.4f}")

In [ ]:
fig = plot_reliability_comparison({
    'NAM': nam_cal['reliability_data'],
    'XGBoost': xgb_cal['reliability_data'],
})
plt.show()

## 5. Prediction Set Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for alpha_idx, alpha in enumerate(alpha_levels[:1]):  # Show alpha=0.05
    nam_sizes = nam_results['y_sets'][:, :, alpha_idx].sum(axis=1)
    xgb_sizes = xgb_results['y_sets'][:, :, alpha_idx].sum(axis=1)
    
    plot_conformal_set_sizes(nam_sizes, alpha, 'NAM', ax=axes[0])
    plot_conformal_set_sizes(xgb_sizes, alpha, 'XGBoost', ax=axes[1])

plt.tight_layout()
plt.show()